# Customer Churn Analysis & Prediction

**Synthetic dataset shaped like Kaggle Telco Customer Churn**

This notebook walks through data loading, cleaning, EDA, modeling, and interpretation. Replace `data/raw_telco_synthetic.csv` with the official Kaggle dataset if you prefer.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
import shap
sns.set(style='whitegrid')

In [ ]:
# Load dataset
df = pd.read_csv("data/raw_telco_synthetic.csv")
df.head()

In [ ]:
# Quick info
df.info()
print("Rows:", len(df))
print("Churn rate:", df['Churn'].value_counts(normalize=True).to_dict())

In [ ]:
# Basic cleaning & feature preparation
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Convert binary Yes/No to 1/0 for Churn
df['ChurnFlag'] = df['Churn'].map({'Yes':1, 'No':0})

# Tenure buckets
bins = [ -1, 6, 12, 24, 48, 72 ]
labels = ['0-6','7-12','13-24','25-48','49-72']
df['tenure_bucket'] = pd.cut(df['tenure'], bins=bins, labels=labels)

df.head()

In [ ]:
# EDA: churn rate by contract
display(df.groupby('Contract')['ChurnFlag'].mean().sort_values(ascending=False).to_frame('churn_rate'))

In [ ]:
# Encode categorical variables
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID','Churn']]
df_enc = pd.get_dummies(df.drop(columns=['customerID','Churn']), columns=cat_cols, drop_first=True)

# Train/test split
X = df_enc.drop('ChurnFlag', axis=1)
y = df_enc['ChurnFlag']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Baseline logistic regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
proba = lr.predict_proba(X_test)[:,1]
print("LR ROC-AUC:", roc_auc_score(y_test, proba))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print(classification_report(y_test, pred))
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
display(importances)

In [ ]:
# SHAP summary plot (may take time on large sets)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
# Uncomment the next line to view plot in notebook
# shap.summary_plot(shap_values[1], X_test)

## Business translation

- Target top 10% customers by predicted churn probability for retention campaigns.
- Estimate expected churn prevented and perform A/B testing to validate interventions.

---

*Notes:* This notebook uses a synthetic dataset. To use the original Kaggle dataset, download `Telco Customer Churn` from Kaggle and replace `data/raw_telco_synthetic.csv` with the downloaded CSV (rename to `raw_telco.csv`). Update the path in the notebook accordingly.